# 05 - Model Training

This notebook focuses on training machine learning models using the engineered student dropout dataset.

The workflow includes:
- Loading the engineered dataset
- Separating features and target
- Splitting the dataset into training and testing sets
- Applying encoding and feature scaling
- Training multiple classification models
- Comparing the initial model performance
- Selecting the best-performing model for further evaluation

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Load Engineered Dataset

In [2]:
df = pd.read_csv("../data/processed/student_dropout_engineered.csv")

df.head()

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,...,Semester,Department,Parental_Education,Dropout,Academic_Performance_Score,Study_Attendance_Score,Stress_Level,Log_Family_Income,Travel_Study_Ratio,Assignment_Delay_Level
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,...,Year 1,Arts,High School,0,0.920000,2.89296,Medium,10.126671,0.101190,Moderate
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,...,Year 3,Engineering,Bachelor,1,1.223333,2.92400,Medium,10.126671,0.170543,Moderate
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,...,Year 1,Arts,Master,0,1.440000,3.11960,Medium,10.601224,0.185227,Low
3,24.4,Male,29740.5,Yes,4.00,82.2,2,38.6,No,No,...,Year 1,CS,High School,1,1.773333,3.28800,Medium,10.300299,0.160833,Moderate
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,...,Year 4,Business,Bachelor,0,1.086667,3.17183,Medium,10.139350,0.091488,Low


In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (10000, 24)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         10000 non-null  float64
 1   Gender                      10000 non-null  object 
 2   Family_Income               10000 non-null  float64
 3   Internet_Access             10000 non-null  object 
 4   Study_Hours_per_Day         10000 non-null  float64
 5   Attendance_Rate             10000 non-null  float64
 6   Assignment_Delay_Days       10000 non-null  int64  
 7   Travel_Time_Minutes         10000 non-null  float64
 8   Part_Time_Job               10000 non-null  object 
 9   Scholarship                 10000 non-null  object 
 10  Stress_Index                10000 non-null  float64
 11  GPA                         10000 non-null  float64
 12  Semester_GPA                10000 non-null  float64
 13  CGPA                        1000

## Separate X and y

In [5]:
X = df.drop(columns=["Dropout"])
y = df["Dropout"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (10000, 23)
Target shape: (10000,)


In [6]:
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

Dropout
0    7646
1    2354
Name: count, dtype: int64
Dropout
0    76.46
1    23.54
Name: proportion, dtype: float64


## Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (8000, 23)
Testing set: (2000, 23)


## Preprocessing Pipeline

In [8]:
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['Age', 'Family_Income', 'Study_Hours_per_Day', 'Attendance_Rate', 'Assignment_Delay_Days', 'Travel_Time_Minutes', 'Stress_Index', 'GPA', 'Semester_GPA', 'CGPA', 'Academic_Performance_Score', 'Study_Attendance_Score', 'Log_Family_Income', 'Travel_Study_Ratio']

Categorical columns:
['Gender', 'Internet_Access', 'Part_Time_Job', 'Scholarship', 'Semester', 'Department', 'Parental_Education', 'Stress_Level', 'Assignment_Delay_Level']


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_columns
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_columns
        )
    ]
)

## Define Models

In [10]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

## Train Models

In [11]:
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    print(f"{name} trained successfully.")

Logistic Regression trained successfully.
Decision Tree trained successfully.
Random Forest trained successfully.
Gradient Boosting trained successfully.


## Initial Model Comparison

In [12]:
results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=1),
        "Recall": recall_score(y_test, y_pred, pos_label=1),
        "F1-Score": f1_score(y_test, y_pred, pos_label=1)
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="F1-Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.7470,0.476882,0.766454,0.587948
3,Gradient Boosting,0.8055,0.647482,0.382166,0.480641
2,Random Forest,0.7995,0.647059,0.326964,0.434415
1,Decision Tree,0.7160,0.403194,0.428875,0.415638


## Best Model

In [13]:
best_model_name = results_df.loc[
    results_df["F1-Score"].idxmax(),
    "Model"
]

best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

Best model: Logistic Regression
